# Démo Interactive — Prédiction de Sentiment en Temps Réel
Utilise le modèle CamemBERT fine-tuné pour prédire le sentiment d'un avis en direct.

In [1]:
# ═══════════════════════════════════════════════════════
# SETUP + CHARGER LE MODÈLE
# ═══════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

!pip install -q gradio transformers

import os
import torch
import gradio as gr
from transformers import CamembertTokenizer, CamembertForSequenceClassification

PROJECT_DIR = '/content/drive/MyDrive/Projet_Sentiment_Analysis'
MODEL_DIR   = os.path.join(PROJECT_DIR, 'models', 'best_camembert')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Charger le modèle
tokenizer = CamembertTokenizer.from_pretrained(MODEL_DIR)
model = CamembertForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(DEVICE)
model.eval()

print(f"✅ Modèle chargé sur {DEVICE}")


Mounted at /content/drive


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Modèle chargé sur cpu


In [2]:
# ═══════════════════════════════════════════════════════
# INTERFACE GRADIO
# ═══════════════════════════════════════════════════════
LABEL_NAMES = ['négatif', 'neutre', 'positif']
EMOJIS      = ['😡', '😐', '😊']

def predire_sentiment(texte):
    """Prédit le sentiment d'un texte."""
    if not texte or len(texte.strip()) < 3:
        return {name: 0.0 for name in LABEL_NAMES}

    encoding = tokenizer(
        texte, truncation=True, padding=True,
        max_length=128, return_tensors='pt'
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**encoding)
        probs = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()

    return {f"{EMOJIS[i]} {name}": float(probs[i]) for i, name in enumerate(LABEL_NAMES)}

# Exemples pré-remplis
exemples = [
    "Application géniale, je l'utilise tous les jours, livraison rapide !",
    "Ça marche pas, l'application plante tout le temps, nul",
    "L'application est correcte, rien de spécial",
    "Service client inexistant, impossible de se faire rembourser",
    "Très bonne app pour les achats en ligne au Maroc",
    "Bof, parfois ça marche parfois non",
]

# Créer l'interface
demo = gr.Interface(
    fn=predire_sentiment,
    inputs=gr.Textbox(
        label="✍️ Écris un avis d'application",
        placeholder="Ex: L'application est super, livraison rapide !",
        lines=3
    ),
    outputs=gr.Label(
        label="📊 Sentiment prédit",
        num_top_classes=3
    ),
    title="🇫🇷 Analyse de Sentiment — Avis d'Apps Marocaines",
    description="Modèle CamemBERT fine-tuné sur 29 000+ avis d'applications marocaines (Glovo, Jumia, Avito...)",
    examples=exemples,
    theme=gr.themes.Soft()
)

demo.launch(share=True)  # share=True crée un lien public

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://144772b0dcc889e399.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
